# 07 — Checkpoint, Sleep, Resume, and Drift

**Chapter 2 teaching lab — §2.7 + Figure 2.6**

> **Teaching outcome:** Govern checkpoint, sleep, resume, drift, and authority decay.

Flow: *concept → why it matters → diagram → read the implementation → expected behavior →
red-team round → playground → challenge.*

In [ ]:
# Setup: make harnesslab/ importable from any working directory, then load this lab's controls.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "harnesslab").is_dir())
sys.path.insert(0, str(ROOT))

from harnesslab import Action, AutonomyLevel, Decision, EXECUTES, Reversibility
from dataclasses import replace
from datetime import datetime, timedelta, timezone
from harnesslab.lifecycle import Checkpoint, Lease, RunState, TRANSITIONS, resume, transition
from harnesslab.viz import decisions, state_path
from harnesslab.playground import lifecycle_playground
from harnesslab.arena import NAIVE, SECURE, run_gauntlet
from harnesslab.viz import gauntlet_board, journey, show_source
from IPython.display import display

journey(7)

## Concept
Long-running agents pause: they wait on humans, sleep until a schedule, or crash and restart. Each
resume is a moment where stale authority can act on a changed world. The control plane treats the run
lifecycle as a **state machine** and re-validates on every resume:

- **Checkpoint integrity**: is this the state we saved, or has it been edited?
- **Authority lease**: has the grant of authority expired while the run slept?
- **Context drift**: has the world the plan assumed changed?
- **Budget**: is there still authority left to spend?

## Why this matters
"It was approved when we started" is the long-horizon version of an unbound approval. Authority should
**decay** unless renewed, and resuming should be a decision, not a default.

## Diagram (Figure 2.6)
```mermaid
stateDiagram-v2
  [*] --> starting
  starting --> running
  running --> paused
  paused --> running
  running --> checkpoint
  checkpoint --> sleeping
  sleeping --> running: seal ok, lease active, same context
  sleeping --> expired: lease expired
  sleeping --> escalated: context drift
  sleeping --> paused: budget exhausted
  expired --> starting: reauthorize
  escalated --> running: drift reviewed
  running --> terminated
  terminated --> [*]
```

## Read the implementation

In [ ]:
show_source(resume, transition)

## Experiment: four ways to wake up

In [ ]:
now = datetime.now(timezone.utc)
cp = Checkpoint("prepare brief", ("search", "draft"), 12, (), ("sources unchanged",), "ctx-v1")
seal = cp.seal()
lease = Lease(now + timedelta(hours=1))
wakeups = {
    "clean resume":    resume(cp, seal, lease, "ctx-v1", now),
    "context drift":   resume(cp, seal, lease, "ctx-v2", now),
    "lease expired":   resume(cp, seal, lease, "ctx-v1", now + timedelta(hours=2)),
    "forged budget":   resume(replace(cp, budget_remaining=999), seal, lease, "ctx-v1", now),
}
colors = {RunState.RUNNING: "allow", RunState.ESCALATED: "require_approval",
          RunState.EXPIRED: "deny", RunState.TERMINATED: "deny", RunState.PAUSED: "stage"}
decisions([(label, colors[state], f"→ {state.value}: {why}") for label, (state, why) in wakeups.items()],
          title="Resume decisions")

## Expected behavior

In [ ]:
assert wakeups["clean resume"][0] is RunState.RUNNING
assert wakeups["context drift"][0] is RunState.ESCALATED
assert wakeups["lease expired"][0] is RunState.EXPIRED
assert wakeups["forged budget"][0] is RunState.TERMINATED

path = [RunState.STARTING]
for nxt in [RunState.RUNNING, RunState.CHECKPOINT, RunState.SLEEPING, RunState.RUNNING, RunState.TERMINATED]:
    path.append(transition(path[-1], nxt))

try:
    transition(RunState.EXPIRED, RunState.RUNNING)
    raise AssertionError("expired run resumed without reauthorization")
except ValueError as e:
    print("blocked:", e)
state_path(path)

## Red-team round (§2.7)
The same attacks run against two defense sets: **HarnessLab** (what this lab built) and
**Shortcuts** (the patterns the chapter warns about). Every attack must be blocked by HarnessLab.
The Shortcuts defense resumes by default: *it was approved when we started*.

In [ ]:
secure, shortcuts = run_gauntlet(SECURE, "2.7"), run_gauntlet(NAIVE, "2.7")
assert all(r.blocked for r in secure), [r.attack.name for r in secure if not r.blocked]
gauntlet_board(secure, shortcuts, title="Gauntlet · §2.7")

## Playground
Let the run sleep, change the world, or tamper with the checkpoint, then see where it wakes.

In [ ]:
lifecycle_playground(cp, lease, now)

## Challenge
Add explicit phases `RESEARCH → STAGE → EXECUTE`. Each phase has its own envelope ceiling, and moving to
a phase with **more** authority requires a checkpoint review (a human-approved checkpoint seal). Show
that a run cannot enter `EXECUTE` with an unreviewed checkpoint, but can drop back to `RESEARCH` at any
time.